# 03 · Train the remote-sensing LoRA on Kaggle

**What this notebook does.** It takes the instruction rows written by `training/vlm/prepare_*.py`
(uploaded to Kaggle as the dataset named in `DATA_SLUG`), fine-tunes a Qwen3-VL-Instruct base with a LoRA
adapter, and pushes the adapter to your Hugging Face account. The AERIS fleet then loads it with
`VLM_ADAPTER_REPOSITORY=<that repo>`.

**Why a LoRA and not full fine-tuning.** A 2B model has ~2 billion weights. Full fine-tuning updates all of
them, needs the optimizer state for all of them (2-3x the weights again in memory) and produces a 4 GB
artefact per experiment. LoRA freezes the base and learns two thin matrices `A` (d×r) and `B` (r×d) per
target layer so that `W' = W + B·A`; with rank r=16 that is ~0.5% of the parameters, trains on a 16 GB T4,
and ships as a ~70 MB file. The base stays exactly the published Apache-2.0 checkpoint, which is also what
lets the fleet serve either variant from one flag.

**Why QLoRA (4-bit base) here.** Kaggle's T4 has 16 GB and no bfloat16. The base is loaded in NF4 (4-bit
NormalFloat, ~1.5 GB for 2B, ~2.6 GB for 4B), the LoRA weights and activations stay in fp16. The adapter
we train on a 4-bit base is served on a 4-bit base in AERIS, so training and inference match.

**What is frozen and why.** The vision tower (SigLIP-style encoder) stays frozen: it was pretrained on
hundreds of millions of images and 30k patches of Lithuania would not improve it, only disturb it. The
language model's attention and MLP projections get LoRA. The vision→language *merger* is left frozen
too, because it is quantised with the base; a later experiment could unfreeze it in fp16 if the boxes
(the most vision-dependent task) lag.

In [ ]:
import os, json, math, random, time, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers>=4.57", "peft>=0.17", "bitsandbytes>=0.45", "accelerate>=1.0", "huggingface_hub"], check=True)

# ---- knobs -----------------------------------------------------------------------------------------
DATA_SLUG = "aeris-vlm-instructions"           # the Kaggle dataset uploaded from data/training/vlm
BASE = os.environ.get("AERIS_VLM_BASE", "Qwen/Qwen3-VL-2B-Instruct")   # or Qwen/Qwen3-VL-4B-Instruct
HF_REPO = os.environ.get("AERIS_HF_REPO", "")   # e.g. "<user>/aeris-qwen3vl-2b-bigearthnet-txt-lora"; empty = save only
IMAGE_PIXELS = 448                              # must equal app/constants/vlm.py VLM_IMAGE_PIXELS
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EPOCHS, LR, BATCH, ACCUMULATE = 1, 2e-4, 2, 8      # effective 16. Batch 4 OOMs a T4: the full-vocabulary logits
                                                    # (151k x ~1,000 tokens x 4, fp32) alone are 2.4 GB; batch 2 peaked at 9.5 GB
MAX_TRAIN_ROWS = int(os.environ.get("AERIS_MAX_ROWS", "0")) or None   # cap for a smoke run
VRSBENCH_IMAGES = int(os.environ.get("AERIS_VRSBENCH_IMAGES", "1500"))   # 0 skips the high-resolution slice
SEED = 7
random.seed(SEED)
DATA = f"/kaggle/input/{DATA_SLUG}"
# A kernel pushed right after `datasets create` can start before Kaggle has finished unpacking the
# dataset (measured: the first push failed on a missing train.jsonl). Wait for the mount, then look for
# the file wherever the unpacking put it.
import glob
for _ in range(60):
    hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
    if hits:
        DATA = os.path.dirname(hits[0]); break
    time.sleep(30)
else:
    raise FileNotFoundError(f"no train.jsonl under /kaggle/input: {os.listdir('/kaggle/input')}")
print(BASE, "->", HF_REPO or "(local only)", "| data at", DATA)

## The data, as the model will see it

Each row is `{images: [...], image_notes: [...], prompt, answer}`. The conversation is built exactly as
`app/models/vlm.py` builds it at inference: the same system prompt, the same note before a SAR image, the
same resize to 448 px on the longer side. **If training and serving disagree on any of these, the adapter
learns a distribution the product never shows it.**

The loss is computed only on the assistant's answer tokens. The prompt tokens get label `-100` (ignored):
we are teaching the model what to *answer*, not to predict the question.

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig

SYSTEM_PROMPT = ("You are a remote-sensing analyst. You read satellite and aerial images: optical (true colour), "
                 "multispectral index maps, and SAR (radar) backscatter. Answer only from what is visible. If the image "
                 "does not show enough to answer, say so.")

def fit_image(path):
    picture = Image.open(path).convert("RGB")
    longer = max(picture.size)
    if longer == IMAGE_PIXELS:
        return picture
    scale = IMAGE_PIXELS / longer
    size = (max(1, round(picture.width * scale)), max(1, round(picture.height * scale)))
    return picture.resize(size, Image.Resampling.BICUBIC if scale > 1 else Image.Resampling.BOX)

def read_rows(name, limit=None):
    rows = [json.loads(l) for l in open(f"{DATA}/{name}", encoding="utf-8") if l.strip()]
    random.shuffle(rows)
    return rows[:limit] if limit else rows

# VRSBench's slice is cut under /tmp, not /kaggle/working: everything under working becomes a kernel
# output, and a few hundred MB of images would be pulled back with the adapter.
IMAGE_ROOTS = {"vrsbench": "/tmp/vrsbench/images"}   # everything else lives in the uploaded dataset

def messages_for(row, with_answer):
    content = []
    root = IMAGE_ROOTS.get(row["source"], f"{DATA}/images")
    for image, note in zip(row["images"], row.get("image_notes") or [None] * len(row["images"])):
        if note:
            content.append({"type": "text", "text": note})
        content.append({"type": "image", "image": fit_image(f"{root}/{image}")})
    content.append({"type": "text", "text": row["prompt"]})
    messages = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
                {"role": "user", "content": content}]
    if with_answer:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": row["answer"]}]})
    return messages

processor = AutoProcessor.from_pretrained(BASE)
# Where the answer starts: the last `<|im_start|>assistant` marker in the token stream. Verified equal to the
# length of a prompt-only pass on real rows; it saves running the image processor twice per row.
ASSISTANT_MARKER = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

def answer_start(ids):
    span = len(ASSISTANT_MARKER)
    for index in range(len(ids) - span, -1, -1):
        if ids[index:index + span] == ASSISTANT_MARKER:
            return index + span
    raise ValueError("no assistant turn in the conversation")

class InstructionDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        full = processor.apply_chat_template(messages_for(row, True), add_generation_prompt=False,
                                             tokenize=True, return_dict=True, return_tensors="pt")
        labels = full["input_ids"].clone()
        labels[:, : answer_start(full["input_ids"][0].tolist())] = -100      # loss on the answer only
        # Token tensors are (1, T) and lose the batch axis; pixel_values is (patches, D) and image_grid_thw
        # (images, 3) are already flat across the conversation and are concatenated as they are.
        item = {k: (v[0] if k in {"input_ids", "attention_mask", "mm_token_type_ids"} else v) for k, v in full.items()}
        item["labels"] = labels[0]
        return item

def collate(items):
    pad = processor.tokenizer.pad_token_id
    longest = max(i["input_ids"].shape[0] for i in items)
    def pad_to(t, value):
        return torch.cat([t, torch.full((longest - t.shape[0],), value, dtype=t.dtype)])
    batch = {
        "input_ids": torch.stack([pad_to(i["input_ids"], pad) for i in items]),
        "attention_mask": torch.stack([pad_to(i["attention_mask"], 0) for i in items]),
        "labels": torch.stack([pad_to(i["labels"], -100) for i in items]),
        "pixel_values": torch.cat([i["pixel_values"] for i in items]),
        "image_grid_thw": torch.cat([i["image_grid_thw"] for i in items]),
    }
    if "mm_token_type_ids" in items[0]:
        batch["mm_token_type_ids"] = torch.stack([pad_to(i["mm_token_type_ids"], 0) for i in items])
    return batch

train_rows = read_rows("train.jsonl", MAX_TRAIN_ROWS)
val_rows = read_rows("validation.jsonl", 200)
if VRSBENCH_IMAGES:
    # The high-resolution slice: VRSBench's 7.8 GB archive is minutes on Kaggle's link and hours on a laptop,
    # so it is cut here. Cartosat-2S at 0.6 m is closer to these 0.1-1 m aerial images than to BigEarthNet's 10 m.
    from huggingface_hub import hf_hub_download
    sys.path.insert(0, DATA)
    from prepare_vrsbench import build_rows as build_vrsbench_rows
    annotations = hf_hub_download("xiang709/VRSBench", "VRSBench_train.json", repo_type="dataset")
    archive = hf_hub_download("xiang709/VRSBench", "Images_train.zip", repo_type="dataset")
    from pathlib import Path
    vrs_rows = build_vrsbench_rows(Path(annotations), Path(archive), Path("/tmp/vrsbench"), VRSBENCH_IMAGES, SEED)
    random.shuffle(vrs_rows)
    train_rows = train_rows + vrs_rows[: (MAX_TRAIN_ROWS // 5 if MAX_TRAIN_ROWS else len(vrs_rows))]
    random.shuffle(train_rows)
    os.remove(archive)
print(len(train_rows), "train rows;", len(val_rows), "validation rows")
from collections import Counter
print(Counter((r["source"], r["type"], r["modality"]) for r in train_rows).most_common(12))

## The model: 4-bit base, LoRA on the language model

`target_modules` is a regex that matches the attention and MLP projections of the **language model only**
(`model.language_model...`), not the vision tower's (`model.visual...`). Matching by name is the usual
mistake here: `q_proj` exists in both towers.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quantisation = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = Qwen3VLForConditionalGeneration.from_pretrained(BASE, dtype=torch.float16,
                                                        quantization_config=quantisation, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

lora = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print("resident MB", torch.cuda.memory_allocated() // 2**20)

## Training

One epoch over ~28k balanced rows (24k of the 42k prepared plus a VRSBench slice - measured on the smoke
run at about one row per second on a T4, so the full 42k would not fit a session). The learning rate
2e-4 with cosine decay and 3% warm-up is the standard QLoRA recipe; higher rates make 4-bit bases
unstable, lower ones need more epochs than a T4 session allows. Effective batch = `BATCH × ACCUMULATE` = 16.

Watch two numbers: the training loss should fall quickly in the first few hundred steps (the model is
learning the answer *formats* - "yes", "c", "[0.1 0.2, 0.3 0.4]") and then slowly (it is learning the
*content*). The validation loss at the end should be below the first evaluation's; if it is not, the
adapter memorised rather than learned, and the fix is fewer rows per template, not more epochs.

In [ ]:
from transformers import Trainer, TrainingArguments

arguments = TrainingArguments(
    output_dir="/kaggle/working/lora", per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUMULATE, num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
    warmup_steps=0.03, weight_decay=0.0, fp16=True, bf16=False, logging_steps=20,   # a float < 1 is a ratio in transformers 5
    eval_strategy="steps", eval_steps=300, save_strategy="steps", save_steps=300, save_total_limit=1,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=4, remove_unused_columns=False, report_to="none", seed=SEED,
    optim="paged_adamw_8bit", max_grad_norm=1.0,
)
from transformers import TrainerCallback

class Heartbeat(TrainerCallback):
    # Kaggle's log shows nothing from the Trainer's own progress bar, so a run looked hung for hours while
    # it was fine. Print step, loss, elapsed and ETA on every log, flushed, so the log tells the truth.
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and state.max_steps:
            elapsed = time.time() - started
            eta = elapsed / max(state.global_step, 1) * (state.max_steps - state.global_step)
            print(f"step {state.global_step}/{state.max_steps}  loss {logs['loss']:.4f}  lr {logs.get('learning_rate', 0):.2e}  "
                  f"elapsed {elapsed/60:.0f} min  eta {eta/60:.0f} min", flush=True)
        if logs and "eval_loss" in logs:
            print(f"step {state.global_step}  validation loss {logs['eval_loss']:.4f}", flush=True)

trainer = Trainer(model=model, args=arguments, train_dataset=InstructionDataset(train_rows),
                  eval_dataset=InstructionDataset(val_rows), data_collator=collate, callbacks=[Heartbeat()])
started = time.time()
print(f"training {len(train_rows)} rows, {len(trainer.get_train_dataloader())} optimizer steps", flush=True)
trainer.train()
print("trained in", round((time.time() - started) / 60), "min; peak MB", torch.cuda.max_memory_allocated() // 2**20)
history = [h for h in trainer.state.log_history if "eval_loss" in h]
print("validation loss:", [(h["step"], round(h["eval_loss"], 4)) for h in history])

## Save and publish the adapter

The adapter directory holds `adapter_config.json` and `adapter_model.safetensors` - tens of MB. A
`training_record.json` beside it states the base, the revision, the data counts and the losses, so the
artefact the fleet loads carries its own provenance. Pushed to the Hub only when `AERIS_HF_REPO` is set
and `HF_TOKEN` is present as a Kaggle secret.

In [ ]:
import shutil
OUT = "/kaggle/working/adapter"
model.save_pretrained(OUT)
processor.save_pretrained(OUT)
shutil.rmtree("/kaggle/working/lora", ignore_errors=True)   # checkpoints are not the deliverable; the adapter is
record = {
    "base": BASE, "image_pixels": IMAGE_PIXELS, "lora": {"rank": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "epochs": EPOCHS, "learning_rate": LR, "effective_batch": BATCH * ACCUMULATE, "train_rows": len(train_rows),
    "validation_loss": [(h["step"], h["eval_loss"]) for h in history], "seed": SEED,
    "data_sources": sorted({r["source"] for r in train_rows}), "trained_on": "kaggle-t4",
}
json.dump(record, open(f"{OUT}/training_record.json", "w"), indent=2)
print(json.dumps(record, indent=2))

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as error:
    print("no HF_TOKEN secret:", error)
if HF_REPO and token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(HF_REPO, private=False, exist_ok=True)
    api.upload_folder(folder_path=OUT, repo_id=HF_REPO, commit_message=f"LoRA on {BASE} from AERIS training run")
    print("pushed to", HF_REPO)